In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

c:\Users\trt\Desktop\research_pyt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 10.69s/it]


In [4]:
# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [5]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

In [6]:
output = generator(prompt)
print(output[0]['generated_text'])

 Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


We can see the model begin to write the email starting with the subject. It stopped
abruptly because it reached the token limit we established by setting max_new_tokens
to 50 tokens. If we increase that, it will continue until concluding the email.

# Choosing a Single Token from the Probability Distribution (Sampling/Decoding)

In [8]:
prompt = "The capital of France is"

In [9]:
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

In [10]:
# Tokenize the input prompt
input_ids = input_ids.to("cuda")

In [11]:
# Get the output of the model before the lm_head
model_output = model.model(input_ids)

In [12]:
# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [13]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

# Parallel Token Processing and Context Size

In [14]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [15]:
lm_head_output.shape

torch.Size([1, 5, 32064])

# Speeding Up Generation by Caching Keys and Values

In [16]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

In [17]:
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [18]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1min ± 1.36 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=False
)

KeyboardInterrupt: 